#### HuggingFace transformers
- input => tokenizer => model => post processing => output
- https://huggingface.co/docs/transformers/v4.52.2/en/main_classes/pipelines

In [1]:
from transformers import pipeline

In [2]:
sentimentClassifier = pipeline("sentiment-analysis")

No model was supplied, defaulted to distilbert/distilbert-base-uncased-finetuned-sst-2-english and revision 714eb0f (https://huggingface.co/distilbert/distilbert-base-uncased-finetuned-sst-2-english).
Using a pipeline without specifying a model name and revision in production is not recommended.
Device set to use cpu


In [3]:
sentimentClassifier("I'm so excited to be learning about large language models")

[{'label': 'POSITIVE', 'score': 0.9997096657752991}]

In [4]:
ner = pipeline("ner", model = "dslim/bert-base-NER")

Some weights of the model checkpoint at dslim/bert-base-NER were not used when initializing BertForTokenClassification: ['bert.pooler.dense.bias', 'bert.pooler.dense.weight']
- This IS expected if you are initializing BertForTokenClassification from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing BertForTokenClassification from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).
Device set to use cpu


In [5]:
ner("Her name is Joey and she works in New York City for Chandler")

[{'entity': 'B-PER',
  'score': 0.99700564,
  'index': 4,
  'word': 'Joey',
  'start': 12,
  'end': 16},
 {'entity': 'B-LOC',
  'score': 0.99954045,
  'index': 9,
  'word': 'New',
  'start': 34,
  'end': 37},
 {'entity': 'I-LOC',
  'score': 0.99931955,
  'index': 10,
  'word': 'York',
  'start': 38,
  'end': 42},
 {'entity': 'I-LOC',
  'score': 0.9995541,
  'index': 11,
  'word': 'City',
  'start': 43,
  'end': 47},
 {'entity': 'B-PER',
  'score': 0.74475205,
  'index': 13,
  'word': 'Chandler',
  'start': 52,
  'end': 60}]

In [6]:
zeroshotClassifier = pipeline("zero-shot-classification", model = "facebook/bart-large-mnli")

Device set to use cpu


In [7]:
sequenceToClassify = "one day I will buy a porsche"
candidateLabels = ["cooking", "programming", "dancing", "driving"]

In [8]:
zeroshotClassifier(sequenceToClassify, candidateLabels)

{'sequence': 'one day I will buy a porsche',
 'labels': ['driving', 'programming', 'dancing', 'cooking'],
 'scores': [0.8680011630058289,
  0.05278439447283745,
  0.04604826867580414,
  0.03316615894436836]}

#### Pre trained tokenizers

In [9]:
from transformers import AutoTokenizer

In [10]:
model = "bert-base-uncased"

In [11]:
tokenizer = AutoTokenizer.from_pretrained(model)

In [12]:
sentence = "I am so excited to be learning about large language models"

In [13]:
inputIds = tokenizer(sentence)
inputIds
# tokentype => distinguish one sent from another 0 => first sentence
# attentionmask => attention to the word

{'input_ids': [101, 1045, 2572, 2061, 7568, 2000, 2022, 4083, 2055, 2312, 2653, 4275, 102], 'token_type_ids': [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]}

In [14]:
tokens = tokenizer.tokenize(sentence)

In [15]:
tokens

['i',
 'am',
 'so',
 'excited',
 'to',
 'be',
 'learning',
 'about',
 'large',
 'language',
 'models']

In [16]:
# predefined vocab specific to the model
tokenIds = tokenizer.convert_tokens_to_ids(tokens)
tokenIds
# some numbers are missing from the input_ids

[1045, 2572, 2061, 7568, 2000, 2022, 4083, 2055, 2312, 2653, 4275]

In [17]:
decodedIds = tokenizer.decode(tokenIds)
decodedIds

'i am so excited to be learning about large language models'

In [18]:
tokenizer.decode(101)

'[CLS]'

In [19]:
tokenizer.decode(102)

'[SEP]'

In [20]:
model2 = "xlnet-base-cased"

In [21]:
tokenizer2 = AutoTokenizer.from_pretrained(model2)

In [22]:
inputIds = tokenizer2(sentence)
inputIds

{'input_ids': [35, 569, 102, 5564, 22, 39, 1899, 75, 392, 1243, 2626, 4, 3], 'token_type_ids': [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 2], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]}

In [23]:
tokens = tokenizer2.tokenize(sentence)
tokens

['▁I',
 '▁am',
 '▁so',
 '▁excited',
 '▁to',
 '▁be',
 '▁learning',
 '▁about',
 '▁large',
 '▁language',
 '▁models']

In [24]:
tokenIds = tokenizer2.convert_tokens_to_ids(tokens)
tokenIds

[35, 569, 102, 5564, 22, 39, 1899, 75, 392, 1243, 2626]

In [25]:
tokenizer2.decode(4) # classification, placed at the beginning

'<sep>'

In [26]:
tokenizer2.decode(3) #seperator of different segments of text

'<cls>'

In [27]:
# Special tokens => specific placeholders or markers that help the model perform various tasks
# Help the model understand the structure and context of the input text
# output of tokenization in a format that the model will understand
# workflow: sentence => tokenize (break words into tokens, add cls and sep tokens and substitute tokens with their ids) => input into the model

In [28]:
# [MASK] => "Fine tuning is [MASK] for all!" => predict, text completion
# task specific tokens => [SOURCE] [TARGET] => guide the model behavior
# [PAD] => extra tokens to make sure all the inputs are the same length
# truncation => consistent inputs

#### HuggingFace and pytorch/tensorflow

In [29]:
from transformers import AutoModelForSequenceClassification
import torch

In [30]:
print(sentence)
print(inputIds)

I am so excited to be learning about large language models
{'input_ids': [35, 569, 102, 5564, 22, 39, 1899, 75, 392, 1243, 2626, 4, 3], 'token_type_ids': [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 2], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]}


In [31]:
tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased-finetuned-sst-2-english")

In [32]:
inputIdsPt = tokenizer(sentence, return_tensors = "pt") #pt ?> return input ids into pytorch tensors
inputIdsPt

{'input_ids': tensor([[ 101, 1045, 2572, 2061, 7568, 2000, 2022, 4083, 2055, 2312, 2653, 4275,
          102]]), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]])}

In [33]:
model = AutoModelForSequenceClassification.from_pretrained("distilbert-base-uncased-finetuned-sst-2-english")

In [34]:
with torch.no_grad():
    logits = model(**inputIdsPt).logits

predictedClassId = logits.argmax().item()
model.config.id2label[predictedClassId]

'POSITIVE'

In [35]:
# save model for the future use => dont need to train it again
modelDir = "saved_models"
tokenizer.save_pretrained(modelDir)

('saved_models\\tokenizer_config.json',
 'saved_models\\special_tokens_map.json',
 'saved_models\\vocab.txt',
 'saved_models\\added_tokens.json',
 'saved_models\\tokenizer.json')

In [ ]:
#myTokenizer = AutoTokenizer.from_pretrained(modelDir)

In [ ]:
#myModel = AutoModelForSequenceClassification.from_pretrained(modelDir)